# vLLM trên Colab — Setup & Serve

Notebook này clone repo dev từ GitHub, chạy script setup (CUDA/torch/vllm) rồi khởi động vLLM OpenAI-compatible server ở background, cuối cùng gửi thử một request để kiểm tra.

Lưu ý: Colab (T4/L4/A100) chỉ dùng để dev/thử nghiệm nhanh. Môi trường chấm điểm chính thức của cuộc thi là NVIDIA H200, Ubuntu 22.04, CUDA 12.x. Cell **1b** cho chọn 1 trong 3 kịch bản CUDA (dev Colab / nộp BTC / test giống BTC trên Colab) — xem chi tiết trong README. Nếu đổi cấu hình CUDA giữa chừng trong cùng session, cell **2** sẽ tự cài rồi tự restart kernel; sau khi thấy kernel restart, chạy lại từ cell 1b.

In [ ]:
# 1. Clone repo (đổi REPO_URL nếu bạn fork/dùng repo khác)
REPO_URL = "https://github.com/Le-Anh-Duy/vLLM-Colab-Inference.git"
REPO_DIR = "colab-setup"

import os
if not os.path.isdir(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !cd "$REPO_DIR" && git pull

In [ ]:
# 1b. Chọn kịch bản CUDA (chỉ set 1 trong 3, để mặc định = kịch bản A)
#
# A. Dev tren Colab (mac dinh, khong can set gi):
#     PINNED_CUDA_WHEEL = "0"  (dung PLAIN_CUDA_VERSION mac dinh = 13.0)
#
# B. Nop bai cho BTC (CUDA/driver da co san tren may cham, khong tu apt install):
#     PINNED_CUDA_WHEEL = "1"; CUDA_VERSION = "12.9"; SKIP_CUDA = "1"
#
# C. Test tren Colab nhung gia lap dung CUDA 12.x nhu BTC:
#     PINNED_CUDA_WHEEL = "1"; CUDA_VERSION = "12.9"
#
# LUU Y: PINNED_CUDA_WHEEL=0 dung bien PLAIN_CUDA_VERSION (mac dinh 13.0),
# PINNED_CUDA_WHEEL=1 dung bien CUDA_VERSION - hai bien nay TACH BIET, doi
# nham bien se khong co tac dung (xem comment trong scripts/setup_vllm.sh).
import os
os.environ.setdefault("PINNED_CUDA_WHEEL", "0")
# Vi du doi sang kich ban C: bo comment 2 dong duoi
# os.environ["PINNED_CUDA_WHEEL"] = "1"
# os.environ["CUDA_VERSION"] = "12.9"

In [ ]:
# 2. Chạy setup (cài CUDA runtime + torch + vllm), tự phát hiện khi cấu hình
# CUDA_VERSION/VLLM_VERSION đổi so với lần chạy trước trong CÙNG session, và
# nếu có đổi thì tự cài xong rồi tự kill kernel để Colab restart (giữ nguyên
# VM/ổ đĩa, chỉ khởi động lại process Python). Sau khi thấy kernel restart,
# CHẠY LẠI CELL NÀY MỘT LẦN NỮA để tiếp tục — lần này sẽ thấy marker khớp và
# bỏ qua cài đặt.
import hashlib
import os
import subprocess
import sys

MARKER_FILE = "/content/.vllm_setup_marker"
tracked_vars = [
    "PINNED_CUDA_WHEEL",
    "CUDA_VERSION",
    "PLAIN_CUDA_VERSION",
    "SKIP_CUDA",
    "SKIP_PYTHON_DEPS",
    "VLLM_VERSION",
]
signature = hashlib.sha256(
    "|".join(f"{k}={os.environ.get(k, '')}" for k in tracked_vars).encode()
).hexdigest()

previous_signature = None
if os.path.exists(MARKER_FILE):
    with open(MARKER_FILE) as f:
        previous_signature = f.read().strip()

if previous_signature == signature:
    print("==> Cau hinh khong doi so voi lan chay truoc, bo qua setup.")
else:
    changed_mid_session = previous_signature is not None

    # Dung Popen + doc stdout theo dong va print() truc tiep, vi subprocess
    # ke thua fd goc (subprocess.call mac dinh) khong luon hien ra trong cell
    # Colab - chi thay traceback ma khong thay log that su xay ra o dau.
    proc = subprocess.Popen(
        ["bash", f"{REPO_DIR}/scripts/setup_vllm.sh"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    output_lines = []
    for line in proc.stdout:
        print(line, end="")
        output_lines.append(line)
    ret = proc.wait()

    if ret != 0:
        tail = "".join(output_lines[-40:])
        raise RuntimeError(
            f"setup_vllm.sh that bai voi exit code {ret}.\n"
            f"--- 40 dong log cuoi ---\n{tail}"
        )

    with open(MARKER_FILE, "w") as f:
        f.write(signature)

    if changed_mid_session:
        print("==> Cau hinh CUDA vua doi trong session nay, dang restart kernel...")
        print("==> Sau khi kernel restart xong, CHAY LAI TU CELL 1b DE TIEP TUC.")
        os.kill(os.getpid(), 9)

In [ ]:
# 3. Khởi động server ở background BẰNG CHÍNH docker/docker-compose.yml (qua
# scripts/serve_from_compose.py) - đảm bảo Colab test đúng 100% flag sẽ nộp
# cho BTC, không còn lệch do copy tay giữa 2 file (từng gây lỗi thật:
# --swap-space/--disable-log-requests đã đổi trong compose nhưng quên sửa
# serve_vllm.sh). Script chỉ thay "--model=/model" bằng MODEL_OVERRIDE (vì
# /model chỉ tồn tại trong container thật của BTC) - mọi flag khác
# (port, served-model-name, max-model-len, kv-cache-dtype...) đọc thẳng từ
# docker-compose.yml.
import subprocess

MODEL_OVERRIDE = "Qwen/Qwen3.5-2B"  # thay cho "/model" (chi ton tai trong container BTC)
COMPOSE_FILE = f"{REPO_DIR}/docker/docker-compose.yml"
PORT = 8000  # co dinh trong docker-compose.yml (dong co comment "Don't change")
SERVED_MODEL_NAME = "Qwen3.5-2B"  # cung co dinh trong docker-compose.yml

!pip install -q pyyaml

# Tu dong phat hien GPU Colab co ho tro FP8 native khong (can SM89+/Hopper
# tro len - vd L4, H100/H200). Neu GPU phien nay (vd T4 SM75, A100 SM80)
# KHONG ho tro, tu dong override --kv-cache-dtype ve "auto" CHI cho lan chay
# nay tren Colab - docker-compose.yml (se nop cho BTC, chay tren H200 SM90)
# KHONG bi doi. Da gap loi that: "FP8 KV cache is not supported ... requires
# SM89+" tren T4.
overrides = []
try:
    import torch
    major, minor = torch.cuda.get_device_capability(0)
    gpu_name = torch.cuda.get_device_name(0)
    supports_fp8 = (major * 10 + minor) >= 89
    print(f"==> GPU Colab: {gpu_name} (compute capability {major}.{minor}) - ho tro FP8 native: {supports_fp8}")
    if not supports_fp8:
        overrides.append("kv-cache-dtype=auto")
except Exception as e:
    print(f"==> Khong detect duoc GPU capability ({e}), bo qua auto-override.")

cmd = [
    "python3", f"{REPO_DIR}/scripts/serve_from_compose.py",
    "--compose-file", COMPOSE_FILE,
    "--model-override", MODEL_OVERRIDE,
]
for ov in overrides:
    cmd += ["--override", ov]

log_file = open("vllm_server.log", "w")
server_proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)
print(f"server_proc.pid = {server_proc.pid}")

In [ ]:
# 4. Chờ server sẵn sàng (poll /health)
import time, urllib.request, urllib.error

url = f"http://localhost:{PORT}/health"

for attempt in range(60):
    try:
        with urllib.request.urlopen(url, timeout=3) as resp:
            if resp.status == 200:
                print("Server san sang.")
                break
    except (urllib.error.URLError, ConnectionError):
        pass
    time.sleep(5)
else:
    print("Server chua san sang, xem log:")
    !tail -n 50 vllm_server.log

In [ ]:
# 5. Test bằng OpenAI client (vllm expose API tương thích OpenAI).
# Goi bang SERVED_MODEL_NAME (khop truong "model" trong request/trace),
# KHONG phai HF repo id (MODEL_OVERRIDE).
!pip install -q openai

from openai import OpenAI

client = OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key="not-needed")
response = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[{"role": "user", "content": "Xin chao, ban co the gioi thieu ban than khong?"}],
    max_tokens=128,
)
print(response.choices[0].message.content)

In [ ]:
# 5b. Do TTFT/TPOT/ERS voi trace THAT cua BTC (scripts/benchmark_traffic.py).
# Nguong Floor/Ceiling/w/gamma mac dinh la gia tri that vong 1 so loai.
TRACE_FILE = f"{REPO_DIR}/baseline-and-input/trace-round1.jsonl"

!pip install -q httpx
!python3 "{REPO_DIR}/scripts/benchmark_traffic.py" \
    --base-url "http://localhost:{PORT}" \
    --model "{SERVED_MODEL_NAME}" \
    --trace-file "{TRACE_FILE}" \
    --output-dir benchmark_results

In [ ]:
# 5c. (Tuy chon) Cham Accuracy Gate bang GPQA Diamond (scripts/eval_gpqa.py).
# BTC chua cong bo 100 cau hoi chinh thuc - can chuan bi file JSONL rieng
# (xem README) hoac dung --hf-fallback de tu test bang bo public tren HF
# (can pip install datasets + quyen truy cap dataset gated).
QUESTIONS_FILE = None  # vd: "gpqa_diamond.jsonl" khi co file that

if QUESTIONS_FILE:
    !python3 "{REPO_DIR}/scripts/eval_gpqa.py" \
        --base-url "http://localhost:{PORT}" \
        --model "{SERVED_MODEL_NAME}" \
        --questions-file "{QUESTIONS_FILE}" \
        --baseline-accuracy 0.4 \
        --output-dir gpqa_results
else:
    print("==> Bo qua: chua set QUESTIONS_FILE. Xem README de biet cach chuan bi file cau hoi.")

In [ ]:
# 6. Dừng server khi xong việc
server_proc.terminate()
server_proc.wait()
print("Da dung server.")